In [ ]:
import sys
import os

# Set working directory explicitly to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

from src.nlp.news_fetcher import NewsFetcher
from src.nlp.sentiment_analyzer import SentimentAnalyzer
from src.data.spark_pipeline import get_spark_session
from src.data.databricks_client import DatabricksClient

print("Phase 2 imports successful!")

In [ ]:
# 1. Fetch news articles
fetcher = NewsFetcher()
news_df = fetcher.fetch_ticker_news("AAPL", limit=10)

# 2. Analyze sentiment
analyzer = SentimentAnalyzer()
sentiment_df = analyzer.add_sentiment_features(news_df)

sentiment_df[["timestamp", "ticker", "title", "compound_score", "pos_score", "neg_score"]].head()

In [ ]:
# Convert sentiment dataset to Spark DataFrame
spark = get_spark_session()
spark_news_df = spark.createDataFrame(sentiment_df)

# Save to Parquet/Delta storage
db_client = DatabricksClient()
db_client.write_dataset(spark_news_df, table_name="aapl_sentiment")

print("Phase 2 Execution Complete!")